# 05 -- WeasyPrint (HTML -> PDF)

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- generer des PDF a partir de HTML + CSS ;
- utiliser Jinja2 pour templater des documents ;
- gerer la mise en page avec `@page` et `@media print` ;
- inclure des images et polices custom ;
- generer des factures, rapports, badges dynamiques ;
- optimiser pour l'impression (marges, sauts de page, headers) ;
- convertir en bytes pour servir via HTTP.

## Prerequis -- ce que vous devez maitriser

- HTML/CSS ;
- Jinja2 ou autre templating ;
- WeasyPrint necessite quelques dependances systeme (Pango, GLib, Cairo).

Ce que ce notebook **ne couvrira pas** :

- JavaScript (WeasyPrint ne l'execute pas) ;
- PDF forms interactifs.

## Plan

1. Installation
2. HTML -> PDF
3. Avec CSS externe
4. Template Jinja2
5. Page setup et marges
6. Headers et footers
7. Sauts de page
8. Tables
9. Images et polices
10. Facture stylee
11. Service web (FastAPI/Flask)
12. Synthese
13. Exercices

---

## 1. Installation

In [ ]:
# pip install weasyprint jinja2
# Debian/Ubuntu : sudo apt install libpango-1.0-0 libpangoft2-1.0-0
from weasyprint import HTML, CSS
import weasyprint
weasyprint.__version__

---

## 2. HTML -> PDF

In [ ]:
import tempfile, pathlib
tmp = pathlib.Path(tempfile.gettempdir())
html = '''<h1>Hello WeasyPrint</h1><p>Premier PDF</p>'''
HTML(string=html).write_pdf(str(tmp / 'hello.pdf'))
print('OK')

---

## 3. Avec CSS externe

In [ ]:
css = '''
h1 { color: steelblue; font-family: sans-serif; }
p { color: #333; line-height: 1.5; }
'''
HTML(string=html).write_pdf(str(tmp / 'styled.pdf'), stylesheets=[CSS(string=css)])
print('OK')

---

## 4. Template Jinja2

In [ ]:
from jinja2 import Template
tpl = Template('''
<h1>Facture {{ numero }}</h1>
<p>Client : {{ client }}</p>
<ul>
{% for item in items %}<li>{{ item.nom }} - {{ item.prix }} EUR</li>{% endfor %}
</ul>
''')
rendered = tpl.render(
    numero='2026-001',
    client='Alice',
    items=[{'nom': 'Pommes', 'prix': 5}, {'nom': 'Pain', 'prix': 2}],
)
HTML(string=rendered).write_pdf(str(tmp / 'facture.pdf'))
print('OK')

---

## 5. Page setup et marges

In [ ]:
page_css = '''
@page {
    size: A4;
    margin: 2cm;
}
body { font-family: sans-serif; }
'''
HTML(string='<h1>Page A4 2cm</h1>').write_pdf(str(tmp / 'page.pdf'), stylesheets=[CSS(string=page_css)])
print('OK')

### Tailles standard

- `A4`, `A3`, `A5`, `letter`, `legal`
- `orientation: landscape`

---

## 6. Headers et footers

In [ ]:
hf_css = '''
@page {
    size: A4;
    margin: 2cm;
    @top-center { content: 'Mon rapport'; font-size: 10pt; }
    @bottom-right { content: 'Page ' counter(page) '/' counter(pages); font-size: 9pt; }
}
'''
HTML(string='<h1>Titre</h1><p>Contenu</p>').write_pdf(str(tmp / 'hf.pdf'), stylesheets=[CSS(string=hf_css)])
print('OK')

---

## 7. Sauts de page

In [ ]:
multi_html = '''
<h1>Chapitre 1</h1>
<p>Contenu...</p>
<div style='page-break-before: always;'></div>
<h1>Chapitre 2</h1>
<p>Suite...</p>
'''
HTML(string=multi_html).write_pdf(str(tmp / 'multi.pdf'))
print('OK')

---

## 8. Tables

In [ ]:
table_html = '''
<style>
table { width: 100%; border-collapse: collapse; }
th, td { border: 1px solid #ccc; padding: 8px; }
th { background: #4a90e2; color: white; }
tr:nth-child(even) { background: #f8f8f8; }
</style>
<table>
<tr><th>Nom</th><th>Age</th></tr>
<tr><td>Alice</td><td>30</td></tr>
<tr><td>Bob</td><td>25</td></tr>
</table>
'''
HTML(string=table_html).write_pdf(str(tmp / 'table.pdf'))
print('OK')

---

## 9. Images et polices

In [ ]:
# Image locale ou en base64
img_html = '''
<img src="https://via.placeholder.com/200" alt="demo">
<p>Avec image</p>
'''
# HTML(string=img_html).write_pdf(str(tmp / 'img.pdf'))
print('Les images sont chargees via HTTP ou chemin local')

In [ ]:
# Polices custom via @font-face
font_css = '''
@font-face {
    font-family: 'Ubuntu';
    src: url('https://example.com/Ubuntu.ttf');
}
body { font-family: Ubuntu, sans-serif; }
'''
print('@font-face fonctionne avec urls accessibles')

---

## 10. Facture stylee

In [ ]:
from jinja2 import Template

tpl = Template('''
<!DOCTYPE html>
<html>
<head>
<style>
@page { size: A4; margin: 2cm; @bottom-right { content: 'Page ' counter(page); } }
body { font-family: sans-serif; color: #333; }
h1 { color: #4a90e2; border-bottom: 2px solid #4a90e2; }
.client { background: #f4f6f9; padding: 12px; border-radius: 4px; }
table { width: 100%; border-collapse: collapse; margin-top: 20px; }
th { background: #4a90e2; color: white; padding: 10px; text-align: left; }
td { padding: 8px; border-bottom: 1px solid #eee; }
.total { font-weight: bold; background: #fffcea; }
</style>
</head>
<body>
<h1>Facture {{ numero }}</h1>
<div class='client'><b>Client :</b> {{ client }}<br><b>Date :</b> {{ date }}</div>
<table>
<tr><th>Article</th><th>Qte</th><th>Prix unit.</th><th>Total</th></tr>
{% for item in items %}
<tr><td>{{ item.nom }}</td><td>{{ item.qte }}</td><td>{{ '%.2f'|format(item.prix) }} EUR</td><td>{{ '%.2f'|format(item.qte * item.prix) }} EUR</td></tr>
{% endfor %}
<tr class='total'><td colspan='3'>Total</td><td>{{ '%.2f'|format(total) }} EUR</td></tr>
</table>
</body>
</html>
''')
items = [
    {'nom': 'Ebook Python', 'qte': 2, 'prix': 29.90},
    {'nom': 'Formation', 'qte': 1, 'prix': 120.00},
]
html_str = tpl.render(
    numero='2026-042',
    client='Alice Dupont',
    date='14 avril 2026',
    items=items,
    total=sum(i['qte'] * i['prix'] for i in items),
)
HTML(string=html_str).write_pdf(str(tmp / 'facture_stylee.pdf'))
print('Facture generee')

---

## 11. Service web (FastAPI/Flask)

In [ ]:
# En FastAPI
# from fastapi import FastAPI
# from fastapi.responses import Response
# app = FastAPI()
# @app.get('/facture/{numero}')
# def facture(numero: int):
#     html_str = render_facture(numero)
#     pdf_bytes = HTML(string=html_str).write_pdf()
#     return Response(pdf_bytes, media_type='application/pdf')
print('write_pdf() sans path retourne bytes')

---

## Synthese

| Operation | WeasyPrint |
|---|---|
| HTML | `HTML(string='...')` ou `HTML(filename='...')` |
| CSS | `CSS(string='...')` ou `CSS(filename='...')` |
| PDF fichier | `.write_pdf(path, stylesheets=[...])` |
| PDF bytes | `.write_pdf()` retourne `bytes` |
| Marges | `@page { margin: 2cm; }` |
| Header | `@page { @top-center { content: '...'; } }` |
| Footer | `@bottom-right { content: 'Page ' counter(page); }` |
| Saut | `page-break-before: always;` |

### Regles a retenir

- **HTML/CSS pour maquetter**, plus rapide que ReportLab pour les documents complexes.
- **Pas de JavaScript** : WeasyPrint ne l'execute pas.
- **CSS 2.1/3 paged media** : `@page`, `@top-left`, etc.
- **Jinja2 + WeasyPrint** = templating efficace pour factures/rapports.
- **Dependances systeme** (Pango, Cairo) obligatoires.

---

## Exercices

### Exercice 1 (facile) -- Hello

Generez un PDF avec un h1 'Hello PDF' et un paragraphe.

In [ ]:
from weasyprint import HTML

# TODO

<details>
<summary>Voir la correction</summary>

```python
from weasyprint import HTML
HTML(string='<h1>Hello PDF</h1><p>Contenu</p>').write_pdf('hello.pdf')
```

</details>

### Exercice 2 (facile) -- CSS externe

Generez un PDF stylise (titre en rouge, texte en bleu).

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from weasyprint import HTML, CSS

css = CSS(string='h1 { color: red; } p { color: blue; }')
HTML(string='<h1>Titre</h1><p>Texte</p>').write_pdf('styled.pdf', stylesheets=[css])
```

</details>

### Exercice 3 (moyen) -- Template de rapport

Fonction `generate_report(title, sections)` ou `sections` est une liste de `(heading, content)`. Genere un PDF avec un h1 titre et pour chaque section un h2 + paragraphe.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from weasyprint import HTML
from jinja2 import Template

tpl = Template('''
<h1>{{ title }}</h1>
{% for h, c in sections %}
<h2>{{ h }}</h2>
<p>{{ c }}</p>
{% endfor %}
''')

def generate_report(title: str, sections: list[tuple[str, str]]) -> bytes:
    html_str = tpl.render(title=title, sections=sections)
    return HTML(string=html_str).write_pdf()
```

</details>

### Exercice 4 (moyen) -- Header/footer avec pagination

PDF avec un contenu multi-page, header 'Rapport' et footer 'Page N'.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from weasyprint import HTML, CSS

css = CSS(string='''
@page {
    margin: 2cm;
    @top-center { content: 'Rapport'; }
    @bottom-right { content: 'Page ' counter(page); }
}
''')

html = '<h1>Page 1</h1><p>...</p>' + '<div style="page-break-before: always;"></div><h1>Page 2</h1>'
HTML(string=html).write_pdf('hf.pdf', stylesheets=[css])
```

</details>

### Exercice 5 (difficile) -- Facture complete

Fonction `generate_invoice(client, items, numero)` qui genere un PDF facture stylise avec table et total.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from weasyprint import HTML
from jinja2 import Template

tpl = Template('''
<style>
body { font-family: sans-serif; }
h1 { color: #4a90e2; }
table { width: 100%; border-collapse: collapse; }
th { background: #4a90e2; color: white; padding: 8px; }
td { padding: 6px; border-bottom: 1px solid #eee; }
.total { font-weight: bold; }
</style>
<h1>Facture {{ numero }}</h1>
<p><b>Client :</b> {{ client }}</p>
<table>
<tr><th>Article</th><th>Qte</th><th>Prix U.</th><th>Total</th></tr>
{% for item in items %}
<tr><td>{{ item.nom }}</td><td>{{ item.qte }}</td><td>{{ '%.2f'|format(item.prix) }}</td><td>{{ '%.2f'|format(item.qte * item.prix) }}</td></tr>
{% endfor %}
<tr class="total"><td colspan="3">Total</td><td>{{ '%.2f'|format(total) }}</td></tr>
</table>
''')

def generate_invoice(client: str, items: list[dict], numero: str) -> bytes:
    total = sum(i['qte'] * i['prix'] for i in items)
    html_str = tpl.render(client=client, items=items, numero=numero, total=total)
    return HTML(string=html_str).write_pdf()
```

</details>

---

## Ressources externes

- [WeasyPrint documentation](https://doc.courtbouillon.org/weasyprint/stable/)
- [CSS Paged Media](https://www.w3.org/TR/css-page-3/)
- [Jinja2 documentation](https://jinja.palletsprojects.com/)